In [0]:
-- ============================================================
-- PIPELINE: Bronze → Silver 
-- ============================================================

INSERT OVERWRITE analisis_tmdb.silver.movies

WITH cleaned_data AS (

    -- 1️ Limpieza básica + conversión de tipos
    SELECT

        id AS movie_id,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(TRIM(title), '^["'']+|["'']+$', ''),
            '^[^A-Za-z0-9]+|[^A-Za-z0-9!?:)]+$',
           ''),
        '\s+',' ') AS title,
        TRIM(original_title) AS original_title,
        TRIM(overview) as overview,
        TRY_CAST(release_date AS DATE) AS release_date,
        UPPER(TRIM(original_language)) as original_language,
        TRY_CAST(popularity AS DECIMAL(10,2)) AS popularity,
        TRY_CAST(vote_average AS DECIMAL(3,2)) AS vote_average,
        TRY_CAST(vote_count AS INT) AS vote_count,
        ingestion_timestamp,
        genre_ids

    FROM analisis_tmdb.bronze.movies_bronze
    WHERE id IS NOT NULL
      AND title IS NOT NULL
      AND release_date IS NOT NULL AND release_date != ''
      AND vote_count >= 0

),

deduplicated AS (

    -- 2 Deduplicación
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY movie_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM cleaned_data

    )
    WHERE rn = 1
),

final_dataset AS (

    -- 3 Métrica derivada
    SELECT

        movie_id,
        title,
        original_title,
        overview,
        release_date,
        YEAR(TRY_CAST(release_date AS DATE)) AS release_year,
        original_language,
        popularity,
        vote_average,
        vote_count,
        ingestion_timestamp

    FROM deduplicated
)

-- 4 Insert final a silver
SELECT *
FROM final_dataset;

In [0]:
-- ============================================================
-- VERIFICACIÓN: Comparar Bronze vs Silver
-- ============================================================

SELECT 
    'Bronze (raw)' as capa,
    COUNT(*) as registros
FROM analisis_tmdb.bronze.movies_bronze

UNION ALL

SELECT 
    'Silver' as capa,
    COUNT(*) as registros
FROM analisis_tmdb.silver.movies;